In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
# read the data
df = pd.read_csv('/kaggle/input/datasets/priyamchoksi/credit-card-transactions-dataset/credit_card_transactions.csv')

# check If worked:
df.head()

**Section 4**

In [ ]:
# Missing Values
print(df.isnull().sum())

# Duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

# Statistics (Transposed)
display(df.describe().T)

**Section 5**

In [ ]:
stats_sumarry = df.describe().T
display(stats_sumarry)

**EDA summary**

In [ ]:
# Summary for city population
print("City Population (city_pop):")
print(df['city_pop'].describe(), "\n")

# Summary for geographic features
geo_features = ['lat', 'long', 'merch_lat', 'merch_long']
print("Geographical Features Summary:")
print(df[geo_features].describe(), "\n")

# Summary for target variable
print("Target Variable (is_fraud):")
print(df['is_fraud'].describe())
print("Fraud Rate:", df['is_fraud'].mean(), "\n")

# Summary for identifier-like features
id_features = ['cc_num', 'zip']
print("Identifier / Location Features Summary:")
print(df[id_features].describe(), "\n")

# Count for merch_zipcode (to show missingness)
print("merch_zipcode non-null count:")
print(df['merch_zipcode'].count())


**Section 6**: How are data distributed

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Find numeric columns
# numeric_cols = df.select_dtypes(include=['number']).columns
# display(numeric_cols)

# 1. Visualizing Fraud vs. Normal
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='is_fraud')
plt.title('Distribution of Fraud (Target Variable)')
plt.show()

# 2. Visualize Transaction Amounts (Numerical)
plt.figure(figsize=(10,5))
# limit the range to 1000
sns.histplot(df[df['amt'] < 1000]['amt'], bins=50, kde=True)
plt.title('Distribution of Transaction Amounts (Filtered < 1000)')
plt.xlabel('Amount ($)')
plt.show()

**Section 7**: Correlation Analysis

In [ ]:
# 1. Define the columns that actually have numerical meaning
meaningful_cols = ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long', 'is_fraud']

# 2. Calculate correlation
clean_corr = df[meaningful_cols].corr()

# 3. Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(clean_corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Analysis (Numeric Features)')
plt.show()

**Section 8**: Outliers

In [ ]:
# Create a boxplot for the transaction amount
# plt.xlabel('Amount ($)')

# Zooming in on transactions under $500 to see the box structure
plt.figure(figsize=(10, 4))
sns.boxplot(x=df[df['amt'] < 500]['amt'], color='orange')
plt.title('Outlier Detection: Transaction Amount (Zoomed < $500)')
plt.show

# Part 2

**Section 9 & 12**: Handling Missing Data & Feature Selection

In [ ]:
# First, remove irreleavant features 
df = df.drop(columns=['first','last','cc_num','street','Unnamed: 0', 'merch_zipcode'])

In [ ]:
df['category_name'] = df['category']

In [ ]:
# Fill numeric missing values
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

In [ ]:
# Fill categorical missing values with 'unknown'
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

In [ ]:
le = LabelEncoder()
high_card_cols = ['merchant', 'category', 'job', 'city', 'state']

for col in high_card_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
print(df[['category', 'category_name']].head())

Clean **Anomalies**

In [ ]:
# Remove outliers in amt
import numpy as np

lower = df['amt'].quantile(0.01)
upper = df['amt'].quantile(0.99)

df['amt'] = np.clip(df['amt'], lower, upper)

**Section 10:** Encoding Categorical Variables

In [ ]:
# Encoding Categorical Variables
le = LabelEncoder()
high_card_cols = ['merchant', 'category', 'job', 'city', 'state']

for col in high_card_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
# Use One-Hot encoding for small categories
df = pd.get_dummies(df, columns=['gender'], drop_first=True)

**Section 11**: Feature Scaling

In [ ]:
# We use RobustScaler because of the extreme outliers in 'amt' [cite: 73, 74]
scaler = RobustScaler()

# List of numeric columns to scale (excluding the target 'is_fraud')
numeric_features = ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long']
df[numeric_features] = scaler.fit_transform(df[numeric_features])

# Part3

Section13: Data Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1. Transaction Amount vs Fraud
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='is_fraud', y='amt')
plt.title('Transaction Amount by Fraud Class')
plt.xlabel('Fraud (0 = No, 1 = Yes)')
plt.ylabel('Scaled Amount')
plt.show()

In [ ]:
# 2. Distribution with overlay
plt.figure(figsize=(10,5))
sns.histplot(data=df, x='amt', hue='is_fraud', bins=50, kde=True)
plt.title('Transaction Amount Distribution by Fraud Class')
plt.xlabel('Scaled Amount')
plt.show()

In [ ]:
# 3. City population vs fraud
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='is_fraud', y='city_pop')
plt.title('City Population by Fraud Class')
plt.show()

In [ ]:
# 4. Fraud by category (top 10)
fraud_by_category = df.groupby('category_name')['is_fraud'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=fraud_by_category.index, y=fraud_by_category.values)
plt.title('Top 10 Categories with Highest Fraud Count')
plt.xlabel('Category')
plt.ylabel('Fraud Count')
plt.xticks(rotation=45)
plt.show()

Section 14: Feature Importance

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Remove target / non-numeric columns
X = df.drop(columns=[
    'is_fraud',
    'category_name',
    'trans_date_trans_time',
    'dob',
    'trans_num'
], errors='ignore')

y = df['is_fraud']



# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Extract importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

display(feature_importance.head(15))

# Plot
plt.figure(figsize=(10,6))
sns.barplot(data=feature_importance.head(15), x='Importance', y='Feature')
plt.title('Top 15 Most Important Features (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()

Section 15: Model Performance Across Features

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(class_weight='balanced', max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
top10 = feature_importance.head(10)['Feature'].tolist()

X_top = df[top10]
y = df['is_fraud']

from sklearn.model_selection import train_test_split

X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(
    X_top, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.ensemble import RandomForestClassifier

rf_top = RandomForestClassifier(n_estimators=100, class_weight='balanced')
rf_top.fit(X_train_top, y_train_top)

y_pred_top = rf_top.predict(X_test_top)
y_prob_top = rf_top.predict_proba(X_test_top)[:, 1]

In [ ]:
#Compare all models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'RF (Top Features)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test_top, y_pred_top)
    ],
    'Precision': [
        precision_score(y_test, y_pred_log, zero_division=0),
        precision_score(y_test, y_pred_rf, zero_division=0),
        precision_score(y_test_top, y_pred_top, zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, y_pred_log, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test_top, y_pred_top, zero_division=0)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_log, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test_top, y_pred_top, zero_division=0)
    ]
})

display(results)